# Inference Demo
**Cross-Lingual Synthetic Augmentation and Language-Aware Training for Code-Mixed Sentiment Analysis**

This notebook demonstrates inference using the saved primary experiment checkpoint.

**Primary Experiment:** Spanish-English | XLM-T | Natural Only  
**Paper F1:** 0.588 | **Reproduced F1:** 0.582

> Run `python train.py` first to generate the checkpoint, or the notebook will fall back to the base XLM-T model.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.dataset import clean_text
from src.model import MODEL_NAMES

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ID2LABEL = {0: 'Positive', 1: 'Negative', 2: 'Neutral'}
LABEL_COLORS = {'Positive': '#2ecc71', 'Negative': '#e74c3c', 'Neutral': '#3498db'}
print(f'Device: {DEVICE}')

## 1. Load Checkpoint

In [ ]:
CHECKPOINT = '../checkpoints/sp_xlmt_natural'

if os.path.exists(CHECKPOINT):
    print(f'Loading fine-tuned checkpoint: {CHECKPOINT}')
    meta_path = os.path.join(CHECKPOINT, 'metadata.json')
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f'  Experiment : {meta["experiment"]}')
        print(f'  Test F1    : {meta["test_f1"]}')
        print(f'  Saved at   : {meta["saved_at"]}')
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT)
else:
    print('No checkpoint found. Loading base XLM-T from HuggingFace.')
    print('Run  python train.py  from the project root first.')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAMES['XLM-T'])
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAMES['XLM-T'], num_labels=3)

model.eval().to(DEVICE)
print('Model ready.')

## 2. Predict Function

In [ ]:
def predict(texts, max_len=64):
    cleaned = [clean_text(t) for t in texts]
    enc = tokenizer(cleaned, max_length=max_len, padding=True,
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        logits = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE)
        ).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()
    results = []
    for i, text in enumerate(texts):
        pred_id = int(probs[i].argmax())
        results.append({
            'text': text,
            'label': ID2LABEL[pred_id],
            'confidence': float(probs[i][pred_id]),
            'scores': {
                'Positive': float(probs[i][0]),
                'Negative': float(probs[i][1]),
                'Neutral':  float(probs[i][2]),
            }
        })
    return results

print('predict() ready')

## 3. Run on Sample Data

In [ ]:
sample_df = pd.read_csv('../data/sample_data.csv')
results = predict(sample_df['text'].tolist())

# Display results
rows = []
for r, true in zip(results, sample_df['label'].tolist()):
    rows.append({
        'Text': r['text'][:45] + '...' if len(r['text']) > 45 else r['text'],
        'True': true,
        'Predicted': r['label'],
        'Confidence': f"{r['confidence']:.1%}",
        'Correct': '✓' if r['label'] == true else '✗'
    })

display_df = pd.DataFrame(rows)
display(display_df.style.applymap(
    lambda v: 'color: green' if v == '✓' else 'color: red',
    subset=['Correct']
))
correct = sum(1 for r in rows if r['Correct'] == '✓')
print(f'\nAccuracy on sample: {correct}/{len(rows)} = {correct/len(rows):.0%}')

## 4. Confidence Score Visualization

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

for i, (r, true) in enumerate(zip(results, sample_df['label'].tolist())):
    ax = axes[i]
    labels_list = ['Positive', 'Negative', 'Neutral']
    scores = [r['scores']['Positive'], r['scores']['Negative'], r['scores']['Neutral']]
    colors = [LABEL_COLORS[l] for l in labels_list]
    bars = ax.bar(labels_list, scores, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability')
    snippet = r['text'][:25] + '...' if len(r['text']) > 25 else r['text']
    match = '✓' if r['label'] == true else '✗'
    ax.set_title(f'{snippet}\nPred: {r["label"]} {match} | True: {true}', fontsize=8)
    ax.tick_params(axis='x', labelsize=8)
    # Highlight predicted bar
    pred_idx = labels_list.index(r['label'])
    bars[pred_idx].set_edgecolor('black')
    bars[pred_idx].set_linewidth(2)

plt.suptitle('Sentiment Prediction Confidence Scores — Sample Data', fontsize=13, fontweight='bold')
plt.tight_layout()
os.makedirs('../results/plots', exist_ok=True)
plt.savefig('../results/plots/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → results/plots/sample_predictions.png')

## 5. Reproduction Results Summary (Table II-IV)

In [ ]:
with open('../results/baseline_metrics.json') as f:
    baseline = json.load(f)

rows = []
for section, exps in baseline.items():
    if section == 'summary': continue
    for exp_name, vals in exps.items():
        rows.append({
            'Experiment': exp_name,
            'Paper F1':   vals['paper_f1'],
            'Repro F1':   vals['repro_f1'],
            'Diff':       vals['diff'],
            'Within 0.04': '✓' if abs(vals['diff']) <= 0.04 else '✗'
        })

rdf = pd.DataFrame(rows)
display(rdf.style
    .background_gradient(subset=['Diff'], cmap='RdYlGn', vmin=-0.12, vmax=0.05)
    .applymap(lambda v: 'color: green' if v == '✓' else 'color: red', subset=['Within 0.04'])
)
within = sum(1 for r in rows if r['Within 0.04'] == '✓')
print(f'\n{within}/13 experiments within 0.04 F1 of paper.')

## 6. Extension Results — Ablation Heatmap

In [ ]:
# Ablation table from paper Table XI
ablation_data = {
    'Standard': [0.7161, 0.7071, 0.7119, 0.6817],
    'CDWL':     [0.7140, 0.7022, 0.7129, 0.6722],
    'LAAF':     [0.7120, 0.7129, 0.7167, 0.6844],
    'CDWL+LAAF':[0.7169, 0.7159, 0.7093, 0.6761],
}
index_labels = ['XLM-T natural', '+Sp-En syn', '+Ma-En syn', 'mBERT natural']

abl_df = pd.DataFrame(ablation_data, index=index_labels)

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(abl_df, annot=True, fmt='.4f', cmap='Blues',
            vmin=0.69, vmax=0.72, ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'})
ax.set_title('Ablation Heatmap: Weighted F1 by Method and Data (Hinglish, XLM-T)',
             fontsize=12, pad=12)
ax.set_xlabel('Method', fontsize=11)
ax.set_ylabel('Data Setup', fontsize=11)
plt.tight_layout()
plt.savefig('../results/plots/ablation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Best: CL1 (CDWL+LAAF, Hinglish natural) = 0.7169')

## 7. Interactive Single Prediction

In [ ]:
# Change these texts to test your own inputs
test_texts = [
    "Me encanta este producto, es increible!",
    "This is absolutely terrible, waste of money.",
    "yaar ye toh theek hai, kuch khas nahi",
    "Lo compré y no me arrepiento para nada.",
    "I don't know if I like it or not.",
]

preds = predict(test_texts)

for p in preds:
    bar = '█' * int(p['confidence'] * 20)
    color_map = {'Positive': '🟢', 'Negative': '🔴', 'Neutral': '🔵'}
    print(f"{color_map[p['label']]} {p['label']:<10} {p['confidence']:.1%}  {bar}")
    print(f"   {p['text']}")
    print()